
# LSMAN: A Lightweight Single-Path Dual-Attention Network for Satellite/Natural Image Super-Resolution

**Author:** Bright Wiredu Nuakoh — AIMS Rwanda MSc Thesis (2023/24)
— **LSMAN** (**L**ightweight
**S**ingle-path **M**ulti-scale **A**ttention **N**etwork) — benchmarked with the *same data protocol and comparison
strategy* used in the two reference papers stored in `papers/`:


## 1. Environment setup

In [ ]:

# Kaggle already ships TensorFlow, tensorflow_datasets, pandas, matplotlib, scikit-image.
# We only need to add the packages that give us pretrained models "for free" (no retraining):
#   - super-image: Hugging-Face-hosted pretrained EDSR / MSRN / CARN (PyTorch, DIV2K-pretrained)
#   - tensorflow_hub: pretrained ESRGAN (DIV2K-pretrained, TensorFlow)
#   - opencv-contrib (dnn_superres module): pretrained FSRCNN / ESPCN (SRCNN's own author lineage)
!pip install -q "super-image" "tensorflow_hub" "datasets>=2.14,<3.0" "huggingface_hub<0.26"
# Kaggle's base image ships plain opencv-python, which does NOT include the `dnn_superres` contrib
# module. Swap it for the contrib build so `cv2.dnn_superres` is importable.
!pip uninstall -y -q opencv-python opencv-python-headless 2>/dev/null
!pip install -q opencv-contrib-python-headless


In [ ]:

import os, io, json, time, zipfile, random, math, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import cv2
from skimage.metrics import structural_similarity as sk_ssim

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Let TensorFlow and PyTorch share the GPU peacefully.
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print('memory growth setting skipped:', e)

print('TensorFlow:', tf.__version__)
print('GPUs visible to TF:', tf.config.list_physical_devices('GPU'))
print('OpenCV:', cv2.__version__, '| has dnn_superres:', hasattr(cv2, 'dnn_superres'))

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| using device:', DEVICE)
if DEVICE.type == 'cpu':
    print('WARNING: no CUDA GPU visible to PyTorch -- EDSR/MSRN/CARN inference will run on CPU and be slow. '
          'On Kaggle: Settings -> Accelerator -> GPU (T4x2 or P100). Locally: this needs an NVIDIA GPU + '
          'a CUDA-enabled PyTorch build (`pip install torch --index-url https://download.pytorch.org/whl/cu121`, '
          'or whichever CUDA version matches your driver) -- the default CPU-only `pip install torch` will not use it.')


## 2. Experiment configuration
Everything you are likely to want to change lives here.

In [ ]:

# --- paths -------------------------------------------------------------
WORK_DIR   = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('./sr_work')
DATA_DIR   = os.path.join(WORK_DIR, 'tfds_data')
WEIGHTS_DIR = os.path.join(WORK_DIR, 'weights')
RESULTS_DIR = os.path.join(WORK_DIR, 'results')
QUAL_DIR    = os.path.join(WORK_DIR, 'qualitative')
for d in [WORK_DIR, DATA_DIR, WEIGHTS_DIR, RESULTS_DIR, QUAL_DIR]:
    os.makedirs(d, exist_ok=True)

# --- experiment scope ----------------------------------------------------
# Both reference papers evaluate x2, x3, x4. We default to x2 and x4 to keep a single
# Kaggle session comfortably within its GPU quota; add 3 to the list below to extend.
SCALES = [2, 4]

# --- data protocol (matches Soh&Cho / Dong et al. Sec. "Experiment Settings") ---
PATCH_SIZE  = 48      # LR patch size
BATCH_SIZE  = 16
AUG_FLIP_ROT = True

# --- training budget (small enough for a single free Kaggle GPU session) ---
DEBUG = False                 # True -> tiny smoke-test run to check the whole pipeline end-to-end first
ITERATIONS      = 2000 if DEBUG else 30000     # total optimizer steps per model per scale
STEPS_PER_EPOCH = 50   if DEBUG else 500
EPOCHS          = max(1, ITERATIONS // STEPS_PER_EPOCH)
INITIAL_LR      = 4e-4
LR_DECAY_EVERY  = max(1, EPOCHS // 3)          # halve LR ~3 times over training, like both papers do

# --- proposed model size (kept "lightweight", i.e. << 2M params) ---
LSMAN_FILTERS = 48
LSMAN_BLOCKS  = 6

# --- how many images per benchmark set to evaluate on (None = all) ---
EVAL_LIMIT = 5 if DEBUG else None
# Large Urban100/BSD100 images are cropped to this max side length purely to keep
# evaluation fast and GPU-memory-safe; every model sees an identical crop, so the
# comparison stays fair.
EVAL_MAX_SIDE = 512

print('WORK_DIR =', WORK_DIR)
print('SCALES =', SCALES, '| EPOCHS =', EPOCHS, '| ITERATIONS =', ITERATIONS)


## 3. DIV2K training data pipeline

We reproduce the exact recipe described in both papers' "Experiment Settings" sections:
*DIV2K (800 train images), bicubic degradation, 48×48 LR patches randomly cropped, random flip + 90° rotation
augmentation, batch size 16.*


In [ ]:

def make_div2k_train_ds(scale, patch_size=PATCH_SIZE, batch_size=BATCH_SIZE):
    builder_name = f'div2k/bicubic_x{scale}'
    ds = tfds.load(builder_name, split='train', data_dir=DATA_DIR, download=True)

    def extract(ex):
        return ex['lr'], ex['hr']
    ds = ds.map(extract, num_parallel_calls=tf.data.AUTOTUNE)

    def random_crop(lr, hr):
        lr_shape = tf.shape(lr)
        lr_h, lr_w = lr_shape[0], lr_shape[1]
        x = tf.random.uniform((), 0, lr_w - patch_size + 1, dtype=tf.int32)
        y = tf.random.uniform((), 0, lr_h - patch_size + 1, dtype=tf.int32)
        lr_patch = lr[y:y + patch_size, x:x + patch_size, :]
        hr_patch = hr[y * scale:y * scale + patch_size * scale,
                       x * scale:x * scale + patch_size * scale, :]
        lr_patch.set_shape([patch_size, patch_size, 3])
        hr_patch.set_shape([patch_size * scale, patch_size * scale, 3])
        return lr_patch, hr_patch
    ds = ds.map(random_crop, num_parallel_calls=tf.data.AUTOTUNE)

    if AUG_FLIP_ROT:
        def augment(lr, hr):
            if tf.random.uniform(()) > 0.5:
                lr = tf.image.flip_left_right(lr); hr = tf.image.flip_left_right(hr)
            k = tf.random.uniform((), 0, 4, dtype=tf.int32)
            lr = tf.image.rot90(lr, k); hr = tf.image.rot90(hr, k)
            return lr, hr
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    def to_float(lr, hr):
        return tf.cast(lr, tf.float32) / 255.0, tf.cast(hr, tf.float32) / 255.0
    ds = ds.map(to_float, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.shuffle(512).repeat().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def make_div2k_val_ds(scale, patch_size=PATCH_SIZE, batch_size=BATCH_SIZE, n_batches=20):
    builder_name = f'div2k/bicubic_x{scale}'
    ds = tfds.load(builder_name, split='validation', data_dir=DATA_DIR, download=True)

    def extract(ex):
        return ex['lr'], ex['hr']
    ds = ds.map(extract, num_parallel_calls=tf.data.AUTOTUNE)

    def center_crop(lr, hr):
        lr_shape = tf.shape(lr)
        lr_h, lr_w = lr_shape[0], lr_shape[1]
        x = (lr_w - patch_size) // 2
        y = (lr_h - patch_size) // 2
        lr_patch = lr[y:y + patch_size, x:x + patch_size, :]
        hr_patch = hr[y * scale:y * scale + patch_size * scale,
                       x * scale:x * scale + patch_size * scale, :]
        lr_patch.set_shape([patch_size, patch_size, 3])
        hr_patch.set_shape([patch_size * scale, patch_size * scale, 3])
        return lr_patch, hr_patch
    ds = ds.map(center_crop, num_parallel_calls=tf.data.AUTOTUNE)

    def to_float(lr, hr):
        return tf.cast(lr, tf.float32) / 255.0, tf.cast(hr, tf.float32) / 255.0
    ds = ds.map(to_float, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.take(n_batches * batch_size).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

print('DIV2K pipeline functions ready.')


## 4. Benchmark test sets: Set5 / Set14 / BSD100 / Urban100 (+ DIV2K validation)


In [ ]:

from datasets import load_dataset
from super_image.data import EvalDataset

BENCHMARK_HF_IDS = {
    'Set5':     'eugenesiow/Set5',
    'Set14':    'eugenesiow/Set14',
    'BSD100':   'eugenesiow/BSD100',
    'Urban100': 'eugenesiow/Urban100',
}

def _to_hwc01(t):
    arr = t.numpy() if hasattr(t, 'numpy') else np.asarray(t)
    if arr.ndim == 3 and arr.shape[0] in (1, 3):     # CHW -> HWC
        arr = np.transpose(arr, (1, 2, 0))
    arr = arr.astype(np.float32)
    if arr.max() > 1.5:
        arr = arr / 255.0
    return np.clip(arr, 0.0, 1.0)

def _center_crop_max_side(lr, hr, scale, max_side=EVAL_MAX_SIDE):
    h, w = lr.shape[:2]
    if max(h, w) <= max_side:
        return lr, hr
    new_h, new_w = min(h, max_side), min(w, max_side)
    y0, x0 = (h - new_h) // 2, (w - new_w) // 2
    lr_c = lr[y0:y0 + new_h, x0:x0 + new_w]
    hr_c = hr[y0 * scale:(y0 + new_h) * scale, x0 * scale:(x0 + new_w) * scale]
    return lr_c, hr_c

def load_benchmark_pairs(name, scale, limit=EVAL_LIMIT):
    if name == 'DIV2K-val':
        ds = tfds.load(f'div2k/bicubic_x{scale}', split='validation', data_dir=DATA_DIR, download=True)
        pairs = []
        for i, ex in enumerate(tfds.as_numpy(ds)):
            if limit is not None and i >= limit:
                break
            lr = ex['lr'].astype(np.float32) / 255.0
            hr = ex['hr'].astype(np.float32) / 255.0
            lr, hr = _center_crop_max_side(lr, hr, scale)
            pairs.append((lr, hr))
        return pairs

    hf_id = BENCHMARK_HF_IDS[name]
    hf_ds = load_dataset(hf_id, f'bicubic_x{scale}', split='validation')
    eval_ds = EvalDataset(hf_ds)
    n = len(eval_ds) if limit is None else min(limit, len(eval_ds))
    pairs = []
    for i in range(n):
        lr_t, hr_t = eval_ds[i]
        lr, hr = _to_hwc01(lr_t), _to_hwc01(hr_t)
        lr, hr = _center_crop_max_side(lr, hr, scale)
        pairs.append((lr, hr))
    return pairs

print('Benchmark loader ready:', list(BENCHMARK_HF_IDS.keys()) + ['DIV2K-val'])


## 5. Evaluation metrics

Matlab-style RGB→Y (YCbCr luminance) conversion, border-shaved by the scale factor, PSNR + SSIM 


In [ ]:

def rgb_to_y(img01):
    r, g, b = img01[..., 0], img01[..., 1], img01[..., 2]
    return 16.0 + 65.481 * r + 128.553 * g + 24.966 * b

def shave(img, border):
    if border <= 0:
        return img
    return img[border:-border, border:-border, ...]

def psnr_ssim_y(sr01, hr01, scale):
    sr01 = np.clip(sr01, 0.0, 1.0)
    hr01 = np.clip(hr01, 0.0, 1.0)
    h = min(sr01.shape[0], hr01.shape[0])
    w = min(sr01.shape[1], hr01.shape[1])
    sr01, hr01 = sr01[:h, :w], hr01[:h, :w]

    y_sr, y_hr = rgb_to_y(sr01), rgb_to_y(hr01)
    y_sr, y_hr = shave(y_sr, scale), shave(y_hr, scale)
    if y_sr.size == 0:
        return np.nan, np.nan

    mse = np.mean((y_sr - y_hr) ** 2)
    psnr = 100.0 if mse <= 1e-10 else 10.0 * np.log10((255.0 ** 2) / mse)
    win = min(7, y_hr.shape[0] - (1 - y_hr.shape[0] % 2), y_hr.shape[1] - (1 - y_hr.shape[1] % 2))
    win = max(3, win - (1 - win % 2))
    try:
        ssim_val = sk_ssim(y_hr, y_sr, data_range=255.0, win_size=win)
    except Exception:
        ssim_val = np.nan
    return psnr, ssim_val

# Keras training-time monitoring metrics (on normalized [0,1] RGB, cheaper than the Y-channel metric above)
def keras_psnr(y_true, y_pred):
    return tf.image.psnr(y_true, y_pred, max_val=1.0)

def keras_ssim(y_true, y_pred):
    return tf.image.ssim(y_true, y_pred, max_val=1.0)

print('Metric functions ready.')


## 6. Classical (non-learned) baselines: Bicubic & Lanczos interpolation

In [ ]:

def classical_predict(method):
    def _predict(lr01, scale):
        h, w = lr01.shape[:2]
        sr = tf.image.resize(lr01, [h * scale, w * scale], method=method)
        return np.clip(sr.numpy(), 0.0, 1.0)
    return _predict

predict_bicubic = classical_predict('bicubic')
predict_lanczos = classical_predict('lanczos3')
print('Classical baselines ready: bicubic, lanczos3')


## 7. SRCNN-lineage baseline — pretrained FSRCNN & ESPCN (no retraining, real weights instead of a heuristic)



In [ ]:

CV2_SR_MODEL_URLS = {
    'FSRCNN': {
        2: 'https://raw.githubusercontent.com/Saafke/FSRCNN_Tensorflow/master/models/FSRCNN_x2.pb',
        3: 'https://raw.githubusercontent.com/Saafke/FSRCNN_Tensorflow/master/models/FSRCNN_x3.pb',
        4: 'https://raw.githubusercontent.com/Saafke/FSRCNN_Tensorflow/master/models/FSRCNN_x4.pb',
    },
    'ESPCN': {
        2: 'https://raw.githubusercontent.com/fannymonori/TF-ESPCN/master/export/ESPCN_x2.pb',
        3: 'https://raw.githubusercontent.com/fannymonori/TF-ESPCN/master/export/ESPCN_x3.pb',
        4: 'https://raw.githubusercontent.com/fannymonori/TF-ESPCN/master/export/ESPCN_x4.pb',
    },
}
CV2_SR_DIR = os.path.join(WORK_DIR, 'cv2_sr_models')
os.makedirs(CV2_SR_DIR, exist_ok=True)

_cv2_sr_cache = {}

def get_cv2_sr(name, scale):
    key = (name, scale)
    if key in _cv2_sr_cache:
        return _cv2_sr_cache[key]
    url = CV2_SR_MODEL_URLS.get(name, {}).get(scale)
    if url is None:
        _cv2_sr_cache[key] = None
        return None
    local_path = os.path.join(CV2_SR_DIR, f'{name}_x{scale}.pb')
    try:
        if not os.path.exists(local_path):
            urllib.request.urlretrieve(url, local_path)
        sr_obj = cv2.dnn_superres.DnnSuperResImpl_create()
        sr_obj.readModel(local_path)
        sr_obj.setModel(name.lower(), scale)
        _cv2_sr_cache[key] = sr_obj
        print(f'Loaded pretrained {name} x{scale} from {url}')
    except Exception as e:
        print(f'!! Could not load {name} x{scale}: {e}. '
              f'(If this is a broken link, download the .pb manually and place it at {local_path}.)')
        _cv2_sr_cache[key] = None
    return _cv2_sr_cache[key]

def cv2_sr_predict(name):
    def _predict(lr01, scale):
        sr_obj = get_cv2_sr(name, scale)
        if sr_obj is None:
            return None
        lr_bgr_u8 = cv2.cvtColor((np.clip(lr01, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
        sr_bgr_u8 = sr_obj.upsample(lr_bgr_u8)
        sr_rgb01 = cv2.cvtColor(sr_bgr_u8, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return sr_rgb01
    return _predict

predict_fsrcnn = cv2_sr_predict('FSRCNN')
predict_espcn = cv2_sr_predict('ESPCN')

for _s in SCALES:
    get_cv2_sr('FSRCNN', _s)
    get_cv2_sr('ESPCN', _s)

def count_pb_params(pb_path):
    # Approximate parameter count: sum of all Const tensors in the frozen graph.
    graph_def = tf.compat.v1.GraphDef()
    with tf.io.gfile.GFile(pb_path, 'rb') as f:
        graph_def.ParseFromString(f.read())
    total = 0
    for node in graph_def.node:
        if node.op == 'Const' and 'value' in node.attr:
            shape = [d.size for d in node.attr['value'].tensor.tensor_shape.dim]
            if shape:
                total += int(np.prod(shape))
    return total


In [ ]:

from tensorflow.keras import layers, models, optimizers, callbacks
print('Keras building blocks imported for LSMAN.')


## 8. LSMAN — proposed lightweight dual-attention model

**Dual-Attention Block (DAB):** two 3×3 convs → squeeze-and-excitation channel attention (RCAB-style, from RCAN
2018 / SMSR 2020) → 7×7-conv spatial attention gate (CBAM-style, Woo et al. 2018) → local residual add.

**LSMAN:** shallow feature conv → `LSMAN_BLOCKS` DABs chained in a **single path** (not parallel, following SMSR's
single-path feature-reuse design) → concatenate *every* intermediate block output with the shallow features (global
feature fusion, 1×1 conv bottleneck, as in RDN) → global residual add → sub-pixel (`depth_to_space`) upsampler
(ESPCN-style) → 3×3 reconstruction conv with sigmoid output.



In [ ]:

def se_block(x, reduction=8):
    ch = x.shape[-1]
    s = layers.GlobalAveragePooling2D()(x)
    s = layers.Dense(max(ch // reduction, 4), activation='relu')(s)
    s = layers.Dense(ch, activation='sigmoid')(s)
    s = layers.Reshape((1, 1, ch))(s)
    return layers.Multiply()([x, s])

def spatial_attention_block(x):
    avg = layers.Lambda(lambda t: tf.reduce_mean(t, axis=-1, keepdims=True))(x)
    mx = layers.Lambda(lambda t: tf.reduce_max(t, axis=-1, keepdims=True))(x)
    concat = layers.Concatenate(axis=-1)([avg, mx])
    attn = layers.Conv2D(1, 7, padding='same', activation='sigmoid')(concat)
    return layers.Multiply()([x, attn])

def dual_attention_block(x, filters):
    inp = x
    y = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    y = layers.Conv2D(filters, 3, padding='same')(y)
    y = se_block(y)
    y = spatial_attention_block(y)
    return layers.Add()([inp, y])

def build_lsman(scale, filters=LSMAN_FILTERS, num_blocks=LSMAN_BLOCKS, name='LSMAN'):
    inp = layers.Input(shape=(None, None, 3))
    f0 = layers.Conv2D(filters, 3, padding='same', name='shallow_feat')(inp)

    x = f0
    block_outputs = []
    for i in range(num_blocks):
        x = dual_attention_block(x, filters)
        block_outputs.append(x)

    fused = layers.Concatenate(name='global_feature_fusion_concat')(block_outputs + [f0])
    fused = layers.Conv2D(filters, 1, padding='same', name='global_feature_fusion_1x1')(fused)
    x = layers.Add(name='global_residual')([f0, fused])

    x = layers.Conv2D(filters * (scale ** 2), 3, padding='same', name='upsample_conv')(x)
    x = layers.Lambda(lambda t: tf.nn.depth_to_space(t, scale), name='depth_to_space')(x)
    out = layers.Conv2D(3, 3, padding='same', activation='sigmoid', name='reconstruction')(x)
    return models.Model(inp, out, name=f'{name}_x{scale}')

lsman_preview = build_lsman(4)
n_params = lsman_preview.count_params()
print(lsman_preview.summary())
print(f'\nLSMAN (x4) parameter count: {n_params:,} ({n_params/1e6:.3f} M) — target was < 2M (lightweight threshold used by Soh & Cho).')


## 9. Training harness (used for LSMAN — the only model trained in this notebook)

**Resumable by design.** Kaggle sessions get interrupted (time limits, manual stops, disconnects) — `train_model()`
detects a previous `training_history_{tag}_x{scale}.csv` from an earlier run, works out how many epochs already
completed from its `epoch` column, reloads the most recent weight checkpoint, and continues from
`initial_epoch=<that count>` instead of retraining from scratch. Just re-run the same cell with the same (or a
larger) `epochs=` and it picks up where it left off.


In [ ]:

def lr_schedule(epoch):
    decays = epoch // LR_DECAY_EVERY
    return INITIAL_LR * (0.5 ** decays)

def train_model(model, scale, tag, epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH, resume=True):
    train_ds = make_div2k_train_ds(scale)
    val_ds = make_div2k_val_ds(scale)

    model.compile(optimizer=optimizers.Adam(learning_rate=INITIAL_LR, beta_1=0.9, beta_2=0.999, epsilon=1e-8),
                  loss='mean_absolute_error',
                  metrics=[keras_psnr, keras_ssim])

    csv_path = os.path.join(RESULTS_DIR, f'training_history_{tag}_x{scale}.csv')
    ckpt_path = os.path.join(WEIGHTS_DIR, f'{tag}_x{scale}_best.weights.h5')
    final_path = os.path.join(WEIGHTS_DIR, f'{tag}_x{scale}_final.weights.h5')

    initial_epoch = 0
    if resume and os.path.exists(csv_path):
        try:
            prev_history = pd.read_csv(csv_path)
            if 'epoch' in prev_history.columns and len(prev_history) > 0:
                initial_epoch = int(prev_history['epoch'].max()) + 1
        except Exception as e:
            print(f'[{tag} x{scale}] could not read previous history ({e}); starting from scratch.')

    resumed = False
    if initial_epoch > 0:
        # Prefer "final" (a previous run finished cleanly) over "best" (previous run was interrupted mid-training).
        weights_to_load = final_path if os.path.exists(final_path) else (ckpt_path if os.path.exists(ckpt_path) else None)
        if weights_to_load is not None:
            try:
                model.load_weights(weights_to_load)
                resumed = True
                print(f'[{tag} x{scale}] resuming from epoch {initial_epoch} (weights: {os.path.basename(weights_to_load)})')
            except Exception as e:
                print(f'[{tag} x{scale}] found history but could not load weights ({e}); starting from scratch.')
                initial_epoch = 0
        else:
            print(f'[{tag} x{scale}] found history CSV but no checkpoint weights; starting from scratch.')
            initial_epoch = 0

    if initial_epoch >= epochs:
        print(f'[{tag} x{scale}] already trained for {initial_epoch} epochs (target {epochs}) — nothing to do. '
              f'Pass a larger `epochs=` to keep training further.')
        return None

    cbs = [
        callbacks.LearningRateScheduler(lr_schedule, verbose=0),
        callbacks.CSVLogger(csv_path, append=resumed),
        callbacks.ModelCheckpoint(ckpt_path, monitor='val_keras_psnr', mode='max',
                                   save_best_only=True, save_weights_only=True),
        callbacks.EarlyStopping(monitor='val_keras_psnr', mode='max', patience=max(10, epochs // 4),
                                 restore_best_weights=True),
    ]

    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, initial_epoch=initial_epoch,
                         steps_per_epoch=steps_per_epoch, callbacks=cbs, verbose=2)
    print(f'[{tag} x{scale}] this run took {(time.time()-t0)/60:.1f} min, '
          f'{model.count_params():,} params, history -> {csv_path}')

    # architecture + final weights (on top of the "best" checkpoint already saved above)
    with open(os.path.join(WEIGHTS_DIR, f'{tag}_x{scale}_architecture.json'), 'w') as f:
        f.write(model.to_json())
    model.save_weights(final_path)
    return history


In [ ]:

lsman_models = {}
for scale in SCALES:
    print(f'\n===== Training LSMAN (proposed)  x{scale} =====')
    m = build_lsman(scale)
    train_model(m, scale, tag='lsman', epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH)
    lsman_models[scale] = m


## 10. Pretrained deep-learning baselines (no retraining — matches the thesis's EDSR and the two papers' EDSR/CARN/MSRN comparisons)

Loaded from the Hugging Face Hub via the `super-image` library, all pretrained on DIV2K with the standard bicubic
protocol — directly the models the thesis (EDSR) and Dong et al. (EDSR, CARN, MSRN) compare against.


In [ ]:

from super_image import EdsrModel, MsrnModel, CarnModel

PRETRAINED_IDS = {
    'EDSR': 'eugenesiow/edsr-base',
    'MSRN': 'eugenesiow/msrn',
    'CARN': 'eugenesiow/carn',
}
PRETRAINED_CLASSES = {'EDSR': EdsrModel, 'MSRN': MsrnModel, 'CARN': CarnModel}

_pretrained_cache = {}

def get_pretrained(name, scale):
    key = (name, scale)
    if key in _pretrained_cache:
        return _pretrained_cache[key]
    cls = PRETRAINED_CLASSES[name]
    hf_id = PRETRAINED_IDS[name]
    try:
        model = cls.from_pretrained(hf_id, scale=scale)
        model.eval()
        model.to(DEVICE)          # <-- without this, inference silently runs on CPU even if a GPU is visible
        _pretrained_cache[key] = model
        print(f'Loaded {name} (scale x{scale}) from {hf_id} -> {DEVICE}')
        return model
    except Exception as e:
        print(f'!! Could not load {name} x{scale} ({hf_id}): {e}')
        _pretrained_cache[key] = None
        return None

def torch_model_predict(name):
    def _predict(lr01, scale):
        model = get_pretrained(name, scale)
        if model is None:
            return None
        inp = torch.from_numpy(lr01.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)
        with torch.no_grad():
            pred = model(inp)
        pred = pred.squeeze(0).clamp(0, 1).cpu().numpy().transpose(1, 2, 0)
        return pred
    return _predict

predict_edsr = torch_model_predict('EDSR')
predict_msrn = torch_model_predict('MSRN')
predict_carn = torch_model_predict('CARN')

# Warm-load / sanity-check for every configured scale.
for name in PRETRAINED_IDS:
    for s in SCALES:
        get_pretrained(name, s)


## 11. Pretrained ESRGAN (TensorFlow Hub) — GAN-based baseline, matches the thesis's SRGAN chapter

`captain-pool/esrgan-tf2` is a DIV2K-pretrained **x4-only** ESRGAN (the enhanced, widely-adopted successor of
SRGAN — same generator lineage the thesis's SRGAN chapter discusses). We only evaluate it at scale 4.


In [ ]:

import tensorflow_hub as hub

_esrgan = None
def get_esrgan():
    global _esrgan
    if _esrgan is None:
        try:
            _esrgan = hub.load('https://tfhub.dev/captain-pool/esrgan-tf2/1')
            print('Loaded ESRGAN from TF-Hub.')
        except Exception as e:
            print('!! Could not load ESRGAN from TF-Hub:', e)
            _esrgan = False
    return _esrgan if _esrgan is not False else None

def predict_esrgan(lr01, scale):
    if scale != 4:
        return None
    model = get_esrgan()
    if model is None:
        return None
    inp = tf.cast(lr01 * 255.0, tf.float32)[tf.newaxis, ...]
    sr = model(inp)
    sr = tf.clip_by_value(sr[0], 0, 255) / 255.0
    return sr.numpy()

get_esrgan()


## 12. Unified model registry (name -> predict function, all take `(lr01, scale) -> sr01 or None`)

In [ ]:

def keras_model_predict(models_by_scale):
    def _predict(lr01, scale):
        model = models_by_scale.get(scale)
        if model is None:
            return None
        pred = model.predict(lr01[np.newaxis, ...], verbose=0)[0]
        return np.clip(pred, 0.0, 1.0)
    return _predict

MODEL_REGISTRY = {
    'Bicubic (classical)': predict_bicubic,
    'Lanczos (classical)': predict_lanczos,
    'FSRCNN (pretrained, SRCNN lineage)': predict_fsrcnn,
    'ESPCN (pretrained, SRCNN lineage)': predict_espcn,
    'EDSR (pretrained)': predict_edsr,
    'MSRN (pretrained)': predict_msrn,
    'CARN (pretrained)': predict_carn,
    'ESRGAN (pretrained, x4 only)': predict_esrgan,
    'LSMAN (proposed)': keras_model_predict(lsman_models),
}

def model_param_count(name):
    if name.startswith('LSMAN'):
        m = next(iter(lsman_models.values()), None)
        return m.count_params() if m else np.nan
    if name.startswith(('FSRCNN', 'ESPCN')):
        key = name.split(' ')[0]
        pb_path = os.path.join(CV2_SR_DIR, f'{key}_x{SCALES[0]}.pb')
        return count_pb_params(pb_path) if os.path.exists(pb_path) else np.nan
    if name.startswith(('EDSR', 'MSRN', 'CARN')):
        key = name.split(' ')[0]
        model = get_pretrained(key, SCALES[0])
        return sum(p.numel() for p in model.parameters()) if model is not None else np.nan
    if name.startswith('ESRGAN'):
        model = get_esrgan()
        try:
            return int(sum(np.prod(v.shape) for v in model.trainable_variables))
        except Exception:
            return 16.7e6  # literature-reported approx. param count for ESRGAN's RRDB generator
    return np.nan

print('Registered models:', list(MODEL_REGISTRY.keys()))


## 13. Run the full benchmark: every model × every scale × every dataset

Produces `results_summary.csv` with one row per (model, scale, dataset): mean PSNR, mean SSIM, #images, and the
model's parameter count.


In [ ]:

DATASETS = ['Set5', 'Set14', 'BSD100', 'Urban100', 'DIV2K-val']

rows = []
_bench_cache = {}

def get_pairs_cached(dataset, scale):
    key = (dataset, scale)
    if key not in _bench_cache:
        _bench_cache[key] = load_benchmark_pairs(dataset, scale)
    return _bench_cache[key]

for scale in SCALES:
    for dataset in DATASETS:
        pairs = get_pairs_cached(dataset, scale)
        print(f'-- scale x{scale} | {dataset}: {len(pairs)} image pairs')
        for model_name, predict_fn in MODEL_REGISTRY.items():
            psnrs, ssims, t_ms = [], [], []
            for lr01, hr01 in pairs:
                t0 = time.time()
                try:
                    sr01 = predict_fn(lr01, scale)
                except Exception as e:
                    print(f'   !! {model_name} failed on {dataset} x{scale}: {e}')
                    sr01 = None
                if sr01 is None:
                    continue
                t_ms.append((time.time() - t0) * 1000)
                p, s = psnr_ssim_y(sr01, hr01, scale)
                psnrs.append(p); ssims.append(s)
            if not psnrs:
                continue
            rows.append({
                'model': model_name,
                'scale': scale,
                'dataset': dataset,
                'n_images': len(psnrs),
                'psnr_db': float(np.nanmean(psnrs)),
                'ssim': float(np.nanmean(ssims)),
                'avg_inference_ms': float(np.mean(t_ms)),
                'num_params': model_param_count(model_name),
            })

results_df = pd.DataFrame(rows)
results_df['num_params_millions'] = results_df['num_params'] / 1e6
results_csv_path = os.path.join(RESULTS_DIR, 'results_summary.csv')
results_df.to_csv(results_csv_path, index=False)
print(f'\nSaved {len(results_df)} rows -> {results_csv_path}')
results_df.sort_values(['scale', 'dataset', 'psnr_db'], ascending=[True, True, False])


In [ ]:

# Pivot view: PSNR (dB) per model x dataset, one table per scale -- handy to paste straight into the thesis.
for scale in SCALES:
    print(f'\n=== PSNR (dB), scale x{scale} ===')
    sub = results_df[results_df.scale == scale]
    pivot = sub.pivot_table(index='model', columns='dataset', values='psnr_db')
    display(pivot.round(2))


## 14. Efficiency comparison: PSNR vs. #parameters (the 'lightweight' story)

In [ ]:

scale_for_plot = max(SCALES)
plot_df = (results_df[(results_df.scale == scale_for_plot) & (results_df.dataset == 'Set5')]
           .dropna(subset=['num_params_millions']))

fig, ax = plt.subplots(figsize=(7, 5))
for _, r in plot_df.iterrows():
    ax.scatter(r['num_params_millions'], r['psnr_db'], s=80)
    ax.annotate(r['model'].split(' (')[0], (r['num_params_millions'], r['psnr_db']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xscale('log')
ax.set_xlabel('Parameters (millions, log scale)')
ax.set_ylabel(f'PSNR (dB) on Set5, x{scale_for_plot}')
ax.set_title('Accuracy vs. model size')
ax.grid(alpha=0.3)
fig.tight_layout()
fig_path = os.path.join(RESULTS_DIR, f'efficiency_plot_x{scale_for_plot}.png')
fig.savefig(fig_path, dpi=150)
plt.show()
print('Saved ->', fig_path)


## 15. Qualitative comparison figure (one representative image per benchmark set)

In [ ]:

def make_qualitative_figure(dataset, scale, index=0):
    pairs = get_pairs_cached(dataset, scale)
    if not pairs:
        return
    lr01, hr01 = pairs[min(index, len(pairs) - 1)]

    names = list(MODEL_REGISTRY.keys())
    n = len(names) + 1
    ncols = 4
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    axes[0].imshow(hr01); axes[0].set_title('Ground truth (HR)'); axes[0].axis('off')
    for i, model_name in enumerate(names, start=1):
        sr01 = MODEL_REGISTRY[model_name](lr01, scale)
        ax = axes[i]
        if sr01 is None:
            ax.axis('off'); continue
        ax.imshow(sr01)
        p, s = psnr_ssim_y(sr01, hr01, scale)
        ax.set_title(f'{model_name}\nPSNR {p:.2f} dB | SSIM {s:.3f}', fontsize=9)
        ax.axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'{dataset} — scale x{scale} — comparative analysis', fontsize=13)
    fig.tight_layout()
    out_path = os.path.join(QUAL_DIR, f'comparison_{dataset}_x{scale}.png')
    fig.savefig(out_path, dpi=150)
    plt.show()
    print('Saved ->', out_path)

for scale in SCALES:
    make_qualitative_figure('Set14', scale, index=0)
    make_qualitative_figure('Urban100', scale, index=0)


## 16. Package everything for download

Zips `weights/` (LSMAN weights & architecture for every scale), `results/` (CSV metrics + training curve
+ efficiency plot) and `qualitative/` (comparison figures) into `outputs.zip`. Download this from the Kaggle output
pane and unzip it under `Thesis/results/` locally so we can regenerate figures / update Chapter 5 from the numbers.


In [ ]:

zip_path = os.path.join(WORK_DIR, 'outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [WEIGHTS_DIR, RESULTS_DIR, QUAL_DIR]:
        for root, _, files in os.walk(folder):
            for fname in files:
                fpath = os.path.join(root, fname)
                arcname = os.path.relpath(fpath, WORK_DIR)
                zf.write(fpath, arcname)

print('Packaged ->', zip_path)
print('Contents:')
with zipfile.ZipFile(zip_path) as zf:
    for n in zf.namelist():
        print(' ', n)
